In [4]:
import pandas as pd
import sqlite3

df = pd.read_csv('Sample - Superstore.csv', encoding='ISO-8859-1')
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

conn = sqlite3.connect(':memory:')
df.to_csv('cleaned_superstore.csv', index=False)
df.to_sql('superstore', conn, index=False, if_exists='replace')

schema = pd.read_sql_query("PRAGMA table_info(superstore);", conn)
print(schema[['name', 'type']])

pd.read_sql_query("SELECT * FROM superstore LIMIT 3;", conn)

             name     type
0          Row_ID  INTEGER
1        Order_ID     TEXT
2      Order_Date     TEXT
3       Ship_Date     TEXT
4       Ship_Mode     TEXT
5     Customer_ID     TEXT
6   Customer_Name     TEXT
7         Segment     TEXT
8         Country     TEXT
9            City     TEXT
10          State     TEXT
11    Postal_Code  INTEGER
12         Region     TEXT
13     Product_ID     TEXT
14       Category     TEXT
15   Sub_Category     TEXT
16   Product_Name     TEXT
17          Sales     REAL
18       Quantity  INTEGER
19       Discount     REAL
20         Profit     REAL


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [5]:
query_filters = """
SELECT Order_ID, Customer_Name, Region, Category, Sales
FROM superstore
WHERE Region = 'South'
  AND Category = 'Technology'
  AND Sales > 100;
"""
print("Filtered Records:")
display(pd.read_sql_query(query_filters, conn).head())

query_top_sales = """
SELECT Order_ID, Product_Name, Category, Sales
FROM superstore
ORDER BY Sales DESC
LIMIT 5;
"""
print("\nTop Sales Transactions:")
display(pd.read_sql_query(query_top_sales, conn))

Filtered Records:


,Order_ID,Customer_Name,Region,Category,Sales
0,CA-2014-158274,Robert Marley,South,Technology,503.960
1,CA-2014-158274,Robert Marley,South,Technology,149.950
2,US-2017-100930,Christopher Schild,South,Technology,617.976
3,CA-2014-167850,Andy Gerbode,South,Technology,178.384
4,US-2016-123750,Ross Baird,South,Technology,408.744



Top Sales Transactions:


,Order_ID,Product_Name,Category,Sales
0,CA-2014-145317,Cisco TelePresence System EX90 Videoconferenci...,Technology,22638.480
1,CA-2016-118689,Canon imageCLASS 2200 Advanced Copier,Technology,17499.950
2,CA-2017-140151,Canon imageCLASS 2200 Advanced Copier,Technology,13999.960
3,CA-2017-127180,Canon imageCLASS 2200 Advanced Copier,Technology,11199.968
4,CA-2017-166709,Canon imageCLASS 2200 Advanced Copier,Technology,10499.970


In [6]:
query_aggregations = """
SELECT Category,
       Sub_Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       SUM(Quantity) AS Total_Quantity,
       ROUND(AVG(Profit), 2) AS Average_Profit
FROM superstore
GROUP BY Category, Sub_Category
ORDER BY Total_Sales DESC;
"""
pd.read_sql_query(query_aggregations, conn)

,Category,Sub_Category,Total_Sales,Total_Quantity,Average_Profit
0,Technology,Phones,330007.05,3289,50.07
1,Furniture,Chairs,328449.10,2356,43.10
2,Office Supplies,Storage,223843.61,3158,25.15
3,Furniture,Tables,206965.53,1241,-55.57
4,Office Supplies,Binders,203412.73,5974,19.84
5,Technology,Machines,189238.63,440,29.43
6,Technology,Accessories,167380.32,2976,54.11
7,Technology,Copiers,149528.03,234,817.91
8,Furniture,Bookcases,114880.00,868,-15.23
9,Office Supplies,Appliances,107532.16,1729,38.92


In [7]:
query_top_customers = """
SELECT Customer_ID, Customer_Name, Segment, ROUND(SUM(Sales), 2) AS Lifetime_Value
FROM superstore
GROUP BY Customer_ID, Customer_Name, Segment
ORDER BY Lifetime_Value DESC
LIMIT 5;
"""
print("Top Customers:")
display(pd.read_sql_query(query_top_customers, conn))

query_monthly_trends = """
SELECT SUBSTR(Order_Date, -4) AS Year,
       CASE
          WHEN Order_Date LIKE '%/%' THEN SUBSTR(Order_Date, 1, INSTR(Order_Date, '/') - 1)
          ELSE SUBSTR(Order_Date, 1, 2)
       END AS Month,
       ROUND(SUM(Sales), 2) AS Monthly_Revenue
FROM superstore
GROUP BY Year, Month
ORDER BY Year DESC, CAST(Month AS INTEGER) ASC
LIMIT 12;
"""
print("\nMonthly Performance:")
display(pd.read_sql_query(query_monthly_trends, conn))

Top Customers:


,Customer_ID,Customer_Name,Segment,Lifetime_Value
0,SM-20320,Sean Miller,Home Office,25043.05
1,TC-20980,Tamara Chand,Corporate,19052.22
2,RB-19360,Raymond Buch,Consumer,15117.34
3,TA-21385,Tom Ashbrook,Home Office,14595.62
4,AB-10105,Adrian Barton,Consumer,14473.57



Monthly Performance:


,Year,Month,Monthly_Revenue
0,2017,1,43971.37
1,2017,2,20301.13
2,2017,3,58872.35
3,2017,4,36521.54
4,2017,5,44261.11
5,2017,6,52981.73
6,2017,7,45264.42
7,2017,8,63120.89
8,2017,9,87866.65
9,2017,10,77776.92


In [8]:
query_validation = """
SELECT COUNT(*) AS Total_Rows,
       COUNT(Row_ID) AS Key_Counts,
       SUM(CASE WHEN Order_ID IS NULL THEN 1 ELSE 0 END) AS Null_Orders,
       SUM(CASE WHEN Sales IS NULL OR Sales <= 0 THEN 1 ELSE 0 END) AS Check_Sales_Anomalies
FROM superstore;
"""
pd.read_sql_query(query_validation, conn)

,Total_Rows,Key_Counts,Null_Orders,Check_Sales_Anomalies
0,9994,9994,0,0
